# Apuntadores, Referencias y Arreglos en C++

**Curso:** Programación Avanzada  
**Kernel:** xeus-cling (C++17)

---

En este cuaderno vamos a explorar tres conceptos fundamentales de C++ que están íntimamente relacionados:

- **Apuntadores**: variables que guardan *direcciones* de memoria
- **Referencias**: alias de variables existentes
- **Arreglos**: bloques contiguos de memoria

La idea central que une los tres es simple: **la memoria del computador es un gran arreglo de celdas numeradas**. Entender cómo C++ nos da acceso directo a esas celdas es lo que distingue a este lenguaje de otros de más alto nivel.

---

## Requisitos previos

Antes de comenzar, ejecuta esta celda para cargar las librerías que usaremos en todo el cuaderno:

In [14]:
#include <iostream>
#include <iomanip>
using namespace std;

---

## Parte 1 — La memoria del computador

Cuando declaras una variable en C++, el sistema operativo reserva espacio en la RAM para guardar su valor. Ese espacio tiene:

1. **Un valor** — el dato que almacena (`16`, `3.14`, `'A'`, etc.)
2. **Una dirección** — el número de celda donde vive en memoria

```
  Dirección   │  Valor
  ────────────┼────────
  0x61ff08    │   16       ← aquí vive int x = 16;
  0x61ff0c    │    ?       ← siguiente celda (otro dato)
```

El operador `&` (dirección-de) nos permite ver esa dirección:

In [15]:
{
    int x = 16;

    cout << "Valor de x:     " << x  << endl;
    cout << "Dirección de x: " << &x << endl;
}

Valor de x:     16
Dirección de x: 0x7ffe421657d4


**¿Por qué la dirección parece un número raro?**  
Las direcciones de memoria se expresan en **hexadecimal** (base 16). El prefijo `0x` indica eso. El valor exacto cambia cada vez que corre el programa, porque el sistema operativo decide dónde colocar las variables.

**¿Cuánto espacio ocupa un `int`?**  
En la mayoría de sistemas modernos, 4 bytes. Puedes verificarlo:

In [16]:
{
    int    a = 0;
    double b = 0.0;
    char   c = 'A';

    cout << "sizeof(int)    = " << sizeof(a) << " bytes" << endl;
    cout << "sizeof(double) = " << sizeof(b) << " bytes" << endl;
    cout << "sizeof(char)   = " << sizeof(c) << " byte"  << endl;
}

sizeof(int)    = 4 bytes
sizeof(double) = 8 bytes
sizeof(char)   = 1 byte


---

## Parte 2 — Apuntadores

### 2.1 ¿Qué es un apuntador?

Un **apuntador** es una variable cuyo valor es **una dirección de memoria**.

```
  Variable normal:          Apuntador:
  ┌─────────┐               ┌──────────────┐
  │   16    │  ← int x      │  0x61ff08    │  ← int* p
  └─────────┘               └──────┬───────┘
  dir: 0x61ff08                     │
                                    ▼
                             ┌─────────┐
                             │   16    │  ← int x
                             └─────────┘
```

### 2.2 Cómo se declara

```cpp
tipo* nombre;     // el * indica que es un apuntador a 'tipo'
```

El tipo importa porque C++ necesita saber **cuántos bytes leer** cuando acceda al valor apuntado. Un `int*` lee 4 bytes; un `double*` lee 8.

In [17]:
{
    int  x = 16;
    int* p;          // declaro un apuntador a int (aún no apunta a nada válido)

    p = &x;          // ahora p guarda la dirección de x

    cout << "Dirección de x:       " << &x << endl;
    cout << "Contenido de p:       " << p  << endl;  // deben ser iguales
    cout << "p apunta al valor:    " << *p << endl;  // * desreferencia: 've' a esa dirección
}

Dirección de x:       0x7ffe421657cc
Contenido de p:       0x7ffe421657cc
p apunta al valor:    16


### 2.3 El operador de desreferencia `*`

El operador `*` tiene dos roles según el contexto:

| Contexto | Significado |
|---|---|
| `int* p;` | Declaración — `p` es un apuntador a `int` |
| `*p` en una expresión | Desreferencia — "ve a la dirección guardada en `p` y dame el valor" |

Piénsalo como ir a un locker del gimnasio: `p` es el número de locker, `*p` es **abrir** el locker y usar su contenido.

In [18]:
{
    // Resumen visual del mismo apuntador
    int  x = 16;
    int* p = &x;

    cout << "p   (la dirección que guarda p): " << p  << endl;
    cout << "*p  (el valor en esa dirección): " << *p << endl;
    cout << "&p  (la dirección donde vive p): " << &p << endl; // p también ocupa memoria!
}

p   (la dirección que guarda p): 0x7ffe421657d4
*p  (el valor en esa dirección): 16
&p  (la dirección donde vive p): 0x7ffe421657c8


> **Nota importante:** `p` y `&p` son distintos. El apuntador `p` también es una variable y por tanto también tiene su propia dirección en memoria. Esto permite construir apuntadores a apuntadores (`int**`), tema de sesiones posteriores.

---

### 2.4 Modificar valores a través de un apuntador

Una de las capacidades más poderosas (y peligrosas) de los apuntadores: podemos **escribir** en la variable original sin mencionarla directamente.

In [19]:
{
    int  x = 16;
    int* p = &x;

    cout << "Antes  — x = " << x << endl;

    *p = 18;   // escribimos 18 en la dirección apuntada por p
               // es exactamente lo mismo que escribir: x = 18;

    cout << "Después — x = " << x << endl;  // x cambió aunque no escribimos x = ...
    cout << "Después — *p = " << *p << endl;
}

Antes  — x = 16
Después — x = 18
Después — *p = 18


**¿Por qué funciona esto?**  
`p` contiene la dirección de `x`. Al escribir `*p = 18`, le estamos diciendo al procesador: *"ve a esa celda de memoria y coloca ahí el valor 18"*. Como `x` vive en esa misma celda, la variable `x` ahora vale `18`.

Esto es la base de **paso por referencia con apuntadores** en funciones — lo veremos más adelante.

---

### 2.5 Apuntador nulo

Un apuntador que no apunta a nada válido debe inicializarse a `nullptr`. Desreferenciar un apuntador nulo es un error en tiempo de ejecución (segfault).

In [20]:
{
    int* p = nullptr;   // apuntador nulo: no apunta a ninguna dirección válida

    if (p == nullptr) {
        cout << "p es nulo, no se puede desreferenciar" << endl;
    } else {
        cout << "Valor: " << *p << endl;
    }
}

p es nulo, no se puede desreferenciar


---

## Parte 3 — Referencias

### 3.1 ¿Qué es una referencia?

Una **referencia** es un **alias** — otro nombre para la misma variable. No es una copia, es exactamente la misma celda de memoria con un nombre diferente.

```
  int x = 16;     →   ┌─────────┐
  int& y = x;         │   16    │  ← x  (y apunta aquí también)
                       └─────────┘
                       dir: 0x61ff08
```

La sintaxis usa `&` en la **declaración** (no confundir con el operador dirección-de):

```cpp
int& y = x;   // y es una referencia a x
```

In [21]:
{
    int  x = 16;
    int& y = x;    // y es un alias de x, NO una copia

    cout << "x = " << x << "  &x = " << &x << endl;
    cout << "y = " << y << "  &y = " << &y << endl;
    // Observa: &x y &y son IGUALES — ocupan la misma celda de memoria
}

x = 16  &x = 0x7ffe421657cc
y = 16  &y = 0x7ffe421657cc


**¿Por qué `&x` y `&y` son iguales?**  
Porque `y` no tiene su propio espacio en memoria: es solo otro nombre para el mismo espacio que ocupa `x`. El compilador trata cualquier uso de `y` como si escribieras `x`.

### 3.2 Modificar el original a través de la referencia

In [22]:
{
    int  x = 16;
    int& y = x;

    cout << "Antes  — x = " << x << endl;

    y = 99;   // modificar y es modificar x

    cout << "Después — x = " << x << endl;
    cout << "Después — y = " << y << endl;
}

Antes  — x = 16
Después — x = 99
Después — y = 99


### 3.3 Diferencias clave: Apuntador vs Referencia

| Característica | Apuntador (`int* p`) | Referencia (`int& r`) |
|---|---|---|
| Puede ser nulo | Sí (`nullptr`) | No — debe inicializarse siempre |
| Puede reasignarse | Sí (`p = &otraVar`) | No — es fija tras la inicialización |
| Ocupa memoria propia | Sí (guarda una dirección) | No (es un alias, mismo espacio) |
| Sintaxis para leer el valor | `*p` | `r` (directo, sin operador) |
| Uso típico | Memoria dinámica, arreglos | Paso por referencia en funciones |

La referencia es más segura y cómoda cuando sabes que siempre va a existir la variable original. El apuntador es más flexible pero requiere más cuidado.

In [23]:
{
    // Los apuntadores pueden reasignarse; las referencias no
    int a = 10, b = 20;

    int* p = &a;
    cout << "p apunta a a: *p = " << *p << endl;

    p = &b;           // reasignamos p para que apunte a b
    cout << "p apunta a b: *p = " << *p << endl;

    // Con referencias:
    int& r = a;       // r es alias de a
    r = b;            // esto NO reasigna la referencia: copia el valor de b en a
    cout << "a ahora vale: " << a << endl;  // a cambió, no r
}

p apunta a a: *p = 10
p apunta a b: *p = 20
a ahora vale: 20


---

## Parte 4 — Arreglos

### 4.1 Arreglos en memoria

Un arreglo es un bloque de celdas **contiguas** en memoria, todas del mismo tipo. Si `int` ocupa 4 bytes, un arreglo de 5 `int` ocupa exactamente 20 bytes consecutivos:

```
  arr[0]  arr[1]  arr[2]  arr[3]  arr[4]
  ┌──────┬──────┬──────┬──────┬──────┐
  │  10  │  20  │  30  │  40  │  50  │
  └──────┴──────┴──────┴──────┴──────┘
  0x100  0x104  0x108  0x10C  0x110
  (+0)   (+4)   (+8)   (+12)  (+16)
```

El nombre del arreglo (`arr`) **es un apuntador** al primer elemento.

In [24]:
{
    int arr[5] = {10, 20, 30, 40, 50};

    cout << "arr        = " << arr  << endl;  // dirección del primer elemento
    cout << "&arr[0]    = " << &arr[0] << endl;  // exactamente lo mismo
    cout << "*arr       = " << *arr << endl;  // valor del primer elemento (desreferencia)
}

arr        = 0x7ffe421657b0
&arr[0]    = 0x7ffe421657b0
*arr       = 10


**¿Por qué `arr` y `&arr[0]` son iguales?**  
En C++, el nombre de un arreglo se convierte automáticamente en un apuntador al primer elemento. Esta conversión se llama *array decay* (decaimiento de arreglo).

### 4.2 Aritmética de apuntadores

Cuando sumamos un entero a un apuntador, **no sumamos bytes**: sumamos múltiplos del tamaño del tipo. `arr + 1` avanza 4 bytes (el tamaño de un `int`), no 1 byte.

In [25]:
{
    int arr[5] = {10, 20, 30, 40, 50};

    cout << "arr   (dir primer elem):   " << arr     << endl;
    cout << "arr+1 (dir segundo elem):  " << arr + 1 << endl;
    cout << "arr+2 (dir tercer elem):   " << arr + 2 << endl;
    cout << endl;
    cout << "*arr       = " << *arr       << " (arr[0])" << endl;
    cout << "*(arr+1)   = " << *(arr + 1) << " (arr[1])" << endl;
    cout << "*(arr+2)   = " << *(arr + 2) << " (arr[2])" << endl;
}

arr   (dir primer elem):   0x7ffe421657c0
arr+1 (dir segundo elem):  0x7ffe421657c4
arr+2 (dir tercer elem):   0x7ffe421657c8

*arr       = 10 (arr[0])
*(arr+1)   = 20 (arr[1])
*(arr+2)   = 30 (arr[2])


**La equivalencia fundamental:** para cualquier arreglo, estas expresiones son idénticas:

```cpp
arr[i]   ≡   *(arr + i)
```

El compilador transforma internamente `arr[i]` en `*(arr + i)`. La notación con corchetes es simplemente azúcar sintáctica.

In [26]:
{
    int arr[5] = {10, 20, 30, 40, 50};

    // Ambas formas son equivalentes
    for (int i = 0; i < 5; i++) {
        cout << "arr[" << i << "] = " << arr[i]
             << "  |  *(arr+" << i << ") = " << *(arr + i) << endl;
    }
}

arr[0] = 10  |  *(arr+0) = 10
arr[1] = 20  |  *(arr+1) = 20
arr[2] = 30  |  *(arr+2) = 30
arr[3] = 40  |  *(arr+3) = 40
arr[4] = 50  |  *(arr+4) = 50


### 4.3 Recorrer un arreglo con un apuntador

Podemos usar un apuntador separado para recorrer el arreglo, porque —a diferencia del nombre del arreglo— el apuntador sí puede moverse:

In [27]:
{
    int arr[5] = {10, 20, 30, 40, 50};
    int* p = arr;   // p apunta al primer elemento

    while (p < arr + 5) {      // mientras no hayamos llegado al final
        cout << *p << " ";
        p++;                   // avanza al siguiente entero (4 bytes adelante)
    }
    cout << endl;
}

10 20 30 40 50 


---

## Parte 5 — Comportamiento indefinido: acceso fuera de límites

C++ **no verifica** los límites de un arreglo en tiempo de ejecución. Si accedes a una posición fuera del arreglo, estás leyendo memoria que pertenece a otras variables — o incluso a otras partes del programa.

In [28]:
{
    // Demostramos el efecto de acceder fuera de los límites
    // Las variables x e y están en memoria cerca del arreglo
    int x   = 9000;
    int arr[5] = {10, 20, 30, 40, 50};
    int y   = 15000;

    cout << "arr[0] al arr[4] (dentro del arreglo):" << endl;
    for (int i = 0; i < 5; i++)
        cout << "  arr[" << i << "] = " << arr[i] << endl;

    cout << endl;
    cout << "Fuera de límites (comportamiento indefinido):" << endl;
    cout << "  arr[5]  = " << arr[5]  << "  ← podría ser y, basura, o causar crash" << endl;
    cout << "  arr[-1] = " << arr[-1] << "  ← podría ser x, basura, o causar crash" << endl;
}

arr[0] al arr[4] (dentro del arreglo):
  arr[0] = 10
  arr[1] = 20
  arr[2] = 30
  arr[3] = 40
  arr[4] = 50

Fuera de límites (comportamiento indefinido):
  arr[5]  = 0  ← podría ser y, basura, o causar crash
  arr[-1] = 5  ← podría ser x, basura, o causar crash


> **¿Por qué C++ no lanza un error automáticamente?**  
> Por eficiencia: verificar cada acceso en tiempo de ejecución tendría un costo de rendimiento. C++ confía en el programador. Esta es una de las razones por las que errores como buffer overflow son tan comunes y peligrosos en C/C++.
>
> En código de producción, usa `std::vector` o `std::array` que ofrecen el método `.at(i)` con verificación de límites.

---

## Parte 6 — Relación entre apuntadores y arreglos

### 6.1 Asignar un arreglo a un apuntador

In [29]:
{
    int arr[5] = {10, 20, 30, 40, 50};
    int* p = arr;   // p apunta al primer elemento de arr

    // Ahora podemos usar p como si fuera arr
    cout << "p[0] = " << p[0] << endl;  // mismo que arr[0]
    cout << "p[2] = " << p[2] << endl;  // mismo que arr[2]
    cout << "p[4] = " << p[4] << endl;  // mismo que arr[4]
}

p[0] = 10
p[2] = 30
p[4] = 50


### 6.2 Diferencia clave: arreglo vs apuntador

Aunque se usen de forma similar, hay una diferencia importante:

In [30]:
{
    int arr[5] = {10, 20, 30, 40, 50};
    int* p = arr;

    // sizeof del arreglo = tamaño total en bytes
    cout << "sizeof(arr) = " << sizeof(arr) << " bytes (5 ints × 4 bytes)" << endl;

    // sizeof del apuntador = tamaño de la dirección (8 bytes en 64-bit)
    cout << "sizeof(p)   = " << sizeof(p)   << " bytes (solo una dirección)" << endl;
}

sizeof(arr) = 20 bytes (5 ints × 4 bytes)
sizeof(p)   = 8 bytes (solo una dirección)


Este es el llamado **problema del array decay**: cuando pasas un arreglo a una función, el compilador lo convierte en un apuntador y se pierde la información del tamaño. Por eso las funciones que reciben arreglos generalmente también reciben su tamaño como parámetro separado.

---

## Ejercicios

### Ejercicio 1
Declara dos variables enteras `a = 5` y `b = 10`. Usa **apuntadores** para intercambiar sus valores sin usar una variable temporal directamente sobre `a` o `b`.

In [ ]:
// Tu solución aquí


### Ejercicio 2
Dado el arreglo `int nums[6] = {3, 7, 1, 9, 4, 6}`, usa **aritmética de apuntadores** (sin usar `[]`) para encontrar e imprimir el valor máximo del arreglo.

In [ ]:
// Tu solución aquí


### Ejercicio 3
Declara una variable `double temp = 98.6`. Crea una **referencia** a ella y una función inline que convierta Fahrenheit a Celsius usando solo la referencia (sin tocar `temp` directamente).

*Fórmula: C = (F - 32) × 5/9*

In [ ]:
// Tu solución aquí


---

## Resumen

| Concepto | Símbolo | Qué hace |
|---|---|---|
| Dirección-de | `&x` | Obtiene la dirección de memoria de `x` |
| Apuntador | `int* p` | Variable que guarda una dirección |
| Desreferencia | `*p` | Lee/escribe el valor en la dirección de `p` |
| Referencia | `int& r = x` | Alias: mismo espacio de memoria, otro nombre |
| Array decay | `arr` en expresión | Se convierte automáticamente en `&arr[0]` |
| Aritmética | `p + i` | Avanza `i × sizeof(tipo)` bytes |
| Equivalencia | `arr[i]` ≡ `*(arr+i)` | Los corchetes son azúcar sintáctica |

---
*Programación Avanzada — Universidad Panamericana*